In [1]:
ROLL_NO = 1024170051
NAME = "Vivek Pandey"
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

In [3]:
df, focus, fingerprint = generate_dataset(ROLL_NO)
tickets = df["Ticket_Text"].tolist()
print("Dataset generated successfully!")
print("Number of tickets:", len(tickets))
print("Fingerprint:", fingerprint)
display(df)


Dataset generated successfully!
Number of tickets: 13
Fingerprint: 55529965


,Ticket_ID,Ticket_Text
0,T01,I need Windows 11 by 5.30 p.m. today.
1,T02,Python is not working since yesterday. Can you...
2,T03,The VPN has stopped syncing. It works on anoth...
3,T04,I reinstalled the student portal after restart...
4,T05,It's not the laptop; it's the server.
5,T06,I reinstalled the VPN while uploading the file...
6,T07,Password is not working since yesterday. It wo...
7,T08,The printer is showing an error. It works on a...
8,T09,Python has stopped syncing since yesterday. Th...
9,T10,The e-mail crashes while opening. It works on ...


### Preprocessing pipeline — do not modify

In [4]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [5]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024170051   Name: Vivek Pandey   Fingerprint: 55529965   Tickets: 13

Ticket_ID                                                                   Ticket_Text
      T01                                         I need Windows 11 by 5.30 p.m. today.
      T02                    Python is not working since yesterday. Can you check this?
      T03                    The VPN has stopped syncing. It works on another computer.
      T04 I reinstalled the student portal after restarting the system. This is urgent.
      T05                                         It's not the laptop; it's the server.
      T06          I reinstalled the VPN while uploading the file. It worked yesterday.
      T07                 Password is not working since yesterday. It worked yesterday.
      T08                The printer is showing an error. It works on another computer.
      T09    Python has stopped syncing since yesterday. The same problem occurs again.
      T10               The e-mail c

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [6]:
# Q1. Preprocessing pipeline

processed = [process_ticket(text) for text in df["Ticket_Text"]]

rows = []
for ticket_id, result in zip(df["Ticket_ID"], processed):
    rows.append({
        "Ticket_ID": ticket_id,
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

summary = pd.DataFrame(rows)
total_row = {
    "Ticket_ID": "TOTAL",
    "Sentences": summary["Sentences"].sum(),
    "Tokens": summary["Tokens"].sum(),
    "Punctuation Tokens": summary["Punctuation Tokens"].sum(),
    "Stopwords": summary["Stopwords"].sum(),
    "Unique Tokens": summary["Unique Tokens"].sum(),
    "Final Tokens": summary["Final Tokens"].sum()
}
summary = pd.concat([summary, pd.DataFrame([total_row])], ignore_index=True)
display(summary)

focus_id = FOCUS[0]
focus_index = df.index[df["Ticket_ID"] == focus_id][0]
focus_result = processed[focus_index]
print("Selected ticket:", focus_id)
print(df.loc[focus_index, "Ticket_Text"])
print("\\nActual tokens:", focus_result["tokens"])
print("Punctuation tokens:", focus_result["punct"])
print("Stopwords:", focus_result["stops"])
print("Final tokens:", focus_result["final"])

print("\\nFinal tokens that may not be complete words:")
print(sorted({t for result in processed for t in result["final"]
              if t in {"n't", "'s", "ca", "'re", "'ve", "'ll", "wo", "sha", "d"} or "'" in t}))

,Ticket_ID,Sentences,Tokens,Punctuation Tokens,Stopwords,Unique Tokens,Final Tokens
0,T01,1,9,1,2,9,6
1,T02,2,12,2,5,12,5
2,T03,2,12,2,4,11,6
3,T04,2,14,2,6,12,6
4,T05,1,11,2,5,8,4
5,T06,2,13,2,5,11,6
6,T07,2,11,2,3,9,6
7,T08,2,13,2,5,12,6
8,T09,2,13,2,4,12,7
9,T10,2,12,2,4,11,6


Selected ticket: T05
It's not the laptop; it's the server.
\nActual tokens: ['it', "'s", 'not', 'the', 'laptop', ';', 'it', "'s", 'the', 'server', '.']
Punctuation tokens: [';', '.']
Stopwords: ['it', 'not', 'the', 'it', 'the']
Final tokens: ["'s", 'laptop', "'s", 'server']
\nFinal tokens that may not be complete words:
["'not", "'s"]


**Answer:** Some final tokens such as `n't`, `'s`, `ca`, `wo` or `sha` may appear because `word_tokenize()` splits contractions into separate pieces. The preprocessing function removes stopwords and punctuation, but these contraction fragments are not always listed as stopwords, so they can remain in the final tokens.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [7]:

import re

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work."
]

raw_texts = df["Ticket_Text"].tolist()

patterns = {
    "Words longer than 5 letters": r"\b[A-Za-z]{6,}\b",
    "Numbers": r"(?<![\w.])\d+(?:[.,]\d+)*(?:%?)(?![\w.])",
    "Capitalized words": r"\b[A-Z][a-zA-Z]*\b",
    "Words beginning with a vowel": r"\b[aeiouAEIOU][A-Za-z]*\b"
}

print("REGEX EXTRACTION FROM RAW TICKETS")

for label, pattern in patterns.items():
    print(f"\n{label}:")
    for ticket_id, text in zip(df["Ticket_ID"], raw_texts):
        matches = re.findall(pattern, text)
        print(f"{ticket_id}: {matches}")

print("\nREGEX EXTRACTION FROM TEST_LINES")

for i, text in enumerate(TEST_LINES, start=1):
    print(f"\nTest line {i}: {text}")
    for label, pattern in patterns.items():
        print(f"{label}: {re.findall(pattern, text)}")

email_pattern = (
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
)

url_pattern = r"(?:https?://|www\.)[^\s<>]+"

phone_pattern = (
    r"(?<!\w)(?:\+\d{1,3}[\s-]?)?"
    r"(?:\d[\s-]?){7,14}\d(?!\w)"
)


def replace_contact_details(text):
    text = re.sub(email_pattern, "<EMAIL>", text)

    def replace_url(match):
        url = match.group(0)
        trailing = ""
        while url and url[-1] in ".,;!?":
            trailing = url[-1] + trailing
            url = url[:-1]
        return "<URL>" + trailing

    text = re.sub(url_pattern, replace_url, text)

    text = re.sub(phone_pattern, "<PHONE>", text)

    return text


print("\nREPLACEMENT ON TEST_LINES")

for line in TEST_LINES:
    print("Original:", line)
    print("Replaced:", replace_contact_details(line))
    print()

print("\nREPLACEMENT ON RAW TICKETS")

for ticket_id, line in zip(df["Ticket_ID"], raw_texts):
    print(ticket_id + ":", replace_contact_details(line))


REGEX EXTRACTION FROM RAW TICKETS

Words longer than 5 letters:
T01: ['Windows']
T02: ['Python', 'working', 'yesterday']
T03: ['stopped', 'syncing', 'another', 'computer']
T04: ['reinstalled', 'student', 'portal', 'restarting', 'system', 'urgent']
T05: ['laptop', 'server']
T06: ['reinstalled', 'uploading', 'worked', 'yesterday']
T07: ['Password', 'working', 'yesterday', 'worked', 'yesterday']
T08: ['printer', 'showing', 'another', 'computer']
T09: ['Python', 'stopped', 'syncing', 'yesterday', 'problem', 'occurs']
T10: ['crashes', 'opening', 'another', 'computer']
T11: ['showing']
T12: ['Submitted']
T13: []

Numbers:
T01: ['11', '5.30']
T02: []
T03: []
T04: []
T05: []
T06: []
T07: []
T08: []
T09: []
T10: []
T11: []
T12: []
T13: ['1023', '03']

Capitalized words:
T01: ['I', 'Windows']
T02: ['Python', 'Can']
T03: ['The', 'VPN', 'It']
T04: ['I', 'This']
T05: ['It']
T06: ['I', 'VPN', 'It']
T07: ['Password', 'It']
T08: ['The', 'It']
T09: ['Python', 'The']
T10: ['The', 'It']
T11: ['The', 'WiF

**Answer:** Capitalized-word extraction also captures words that are capitalized only because they begin a sentence. The exact count depends on the generated tickets. The next cell calculates a simple count of capitalized matches that occur at the beginning of a sentence; proper nouns and acronyms may also be capitalized for other reasons.

## Q3. Custom tokenizer
The custom tokenizer uses one regular-expression pattern and keeps URLs, email addresses, phone numbers, versions, decimals, contractions and hyphenated words together.

In [8]:
# Q3. Custom tokenization
from nltk.tokenize import RegexpTokenizer

token_pattern = (
    r"""https?://[^\s<>]+|www\.[^\s<>]+|"""
    r"""[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|"""
    r"""(?:\+\d{1,3}[\s-]?)?(?:\d[\s-]?){7,14}\d|"""
    r"""\d+(?:\.\d+)+%?|\d+(?:,\d{3})+(?:\.\d+)?%?|"""
    r"""[A-Za-z]+(?:['’][A-Za-z]+)+|"""
    r"""[A-Za-z]+(?:-[A-Za-z]+)+|"""
    r"""[A-Za-z]+|"""
    r"""[^\w\s]"""
)
custom_tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return custom_tokenizer.tokenize(text)

print("CUSTOM TOKENIZER ON TEST_LINES")
for line in TEST_LINES:
    print("\\nText:", line)
    print("my_tokenize:", my_tokenize(line))
    print("word_tokenize:", word_tokenize(line))

print("\\nCUSTOM TOKENIZER ON TICKETS")
comparison_rows = []
for ticket_id, text in zip(df["Ticket_ID"], raw_texts):
    custom = my_tokenize(text)
    standard = word_tokenize(text)
    comparison_rows.append({
        "Ticket_ID": ticket_id,
        "Custom Tokens": custom,
        "word_tokenize Tokens": standard,
        "Different": custom != standard
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df[["Ticket_ID", "Different"]])
print("Tickets with different tokenization:",
      int(comparison_df["Different"].sum()), "out of", len(comparison_df))

print("\\nThree common differences:")
print("1. The custom tokenizer keeps contractions such as isn't together; word_tokenize may split them.")
print("2. The custom tokenizer keeps hyphenated forms such as state-of-the-art together.")
print("3. The custom tokenizer keeps URLs, email addresses, phone numbers and version numbers as single tokens.")

CUSTOM TOKENIZER ON TEST_LINES
\nText: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
my_tokenize: ['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', ';', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']
word_tokenize: ['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']
\nText: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
my_tokenize: ['Call', '+91 98765 43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5%', 'done', ';', 'fee', 'Rs', '1,250', '.']
word_tokenize: ['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']
\nText: The state-of-the-art lab isn't open; log-in at 5.30 p.

,Ticket_ID,Different
0,T01,True
1,T02,False
2,T03,False
3,T04,False
4,T05,True
5,T06,False
6,T07,False
7,T08,False
8,T09,False
9,T10,False


Tickets with different tokenization: 3 out of 13
\nThree common differences:
1. The custom tokenizer keeps contractions such as isn't together; word_tokenize may split them.
2. The custom tokenizer keeps hyphenated forms such as state-of-the-art together.
3. The custom tokenizer keeps URLs, email addresses, phone numbers and version numbers as single tokens.


**Answer:** The exact number is printed by the comparison cell. Differences occur because the custom regular expression deliberately keeps selected multi-part items (such as contractions, hyphenated words, URLs, email addresses, phone numbers and versions) together, while `word_tokenize()` follows its own tokenization rules.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [9]:

from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
import pandas as pd

unique_final_tokens = sorted({
    token
    for result in processed
    for token in result["final"]
})

porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged_words = pos_tag(unique_final_tokens)
pos_by_word = {word: tag for word, tag in tagged_words}

candidate_suffixes = ["ing", "ed", "ly", "es", "s", "er"]

suffix = None
matching_words = []

for candidate in candidate_suffixes:
    matches = [
        word for word in unique_final_tokens
        if word.endswith(candidate) and len(word) > len(candidate)
    ]

    if len(matches) >= 2:
        suffix = candidate
        matching_words = matches
        break

if suffix is None:
    raise ValueError(
        "No candidate suffix occurs in at least two unique final tokens. "
        "Inspect unique_final_tokens and choose a suitable suffix."
    )

regexp_stemmer = RegexpStemmer(suffix + r"$", min=4)

print("Selected suffix:", suffix)
print("Dataset words supporting this choice:", matching_words)
print(
    f"The suffix '{suffix}' was selected because it occurs "
    "in at least two unique final tokens in my dataset."
)

stem_rows = []

for word in unique_final_tokens:
    pos = wordnet_pos(pos_by_word.get(word, "NN"))

    outputs = {
        "Word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp": regexp_stemmer.stem(word),
        "POS Lemma": lemmatizer.lemmatize(word, pos=pos)
    }

    results = [
        outputs["Porter"],
        outputs["Lancaster"],
        outputs["Regexp"],
        outputs["POS Lemma"]
    ]

    if len(set(results)) > 1:
        stem_rows.append(outputs)

stem_table = pd.DataFrame(
    stem_rows,
    columns=["Word", "Porter", "Lancaster", "Regexp", "POS Lemma"]
)

print("\nWords for which the outputs differ:")
display(stem_table)

print("\nNumber of words with differing outputs:", len(stem_table))

Selected suffix: ing
Dataset words supporting this choice: ['opening', 'restarting', 'showing', 'syncing', 'uploading', 'working']
The suffix 'ing' was selected because it occurs in at least two unique final tokens in my dataset.

Words for which the outputs differ:


,Word,Porter,Lancaster,Regexp,POS Lemma
0,another,anoth,anoth,another,another
1,computer,comput,comput,computer,computer
2,crashes,crash,crash,crashes,crash
3,error,error,er,error,error
4,file,file,fil,file,file
5,lms,lm,lms,lms,lm
6,need,need,nee,need,need
7,occurs,occur,occ,occurs,occurs
8,opening,open,op,open,open
9,portal,portal,port,portal,portal



Number of words with differing outputs: 26


**Answer:** Over-stemming happens when a stemmer removes too much and produces a form that is not a meaningful word or combines words with different meanings. Under-stemming happens when related forms are not reduced to the same stem. Use the displayed comparison table to identify one actual example from this roll-number-specific dataset.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
# Q5. Corpus statistics
from collections import Counter
from nltk.util import ngrams
import numpy as np
import matplotlib.pyplot as plt

S1 = [token for result in processed for token in result["tokens"]]
S2 = [token for token in S1 if not is_punct(token)]
S3 = [token for result in processed for token in result["final"]]
S4 = [porter.stem(token) for token in S3]

def corpus_stats(tokens):
    return {
        "Total Tokens": len(tokens),
        "Unique Tokens": len(set(tokens)),
        "TTR": round(len(set(tokens)) / len(tokens), 4) if tokens else 0
    }

stats_table = pd.DataFrame({
    "S1: All tokens": corpus_stats(S1),
    "S2: Without punctuation": corpus_stats(S2),
    "S3: Final tokens": corpus_stats(S3),
    "S4: Porter-stemmed S3": corpus_stats(S4)
}).T
display(stats_table)

print("Top 10 words in S1:")
display(pd.DataFrame(Counter(S1).most_common(10), columns=["Word", "Frequency"]))

print("Top 10 words in S3:")
display(pd.DataFrame(Counter(S3).most_common(10), columns=["Word", "Frequency"]))

freq = Counter(S3)
ranked = sorted(freq.values(), reverse=True)
ranks = np.arange(1, len(ranked) + 1)
plt.figure(figsize=(7, 5))
plt.plot(np.log(ranks), np.log(ranked), marker="o", linewidth=1)
plt.xlabel("log(Rank)")
plt.ylabel("log(Frequency)")
plt.title("Log(Rank) vs Log(Frequency) for S3")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 5 bigrams in S1:")
display(pd.DataFrame(Counter(ngrams(S1, 2)).most_common(5),
                     columns=["Bigram", "Frequency"]))

print("Top 5 bigrams in S3:")
display(pd.DataFrame(Counter(ngrams(S3, 2)).most_common(5),
                     columns=["Bigram", "Frequency"]))

print("TTR explanation: TTR is unique tokens divided by total tokens. Removing punctuation and stopwords changes both counts; stemming can merge several different word forms into the same stem, usually reducing the number of unique types and therefore changing TTR.")

,Total Tokens,Unique Tokens,TTR
S1: All tokens,146.0,67.0,0.4589
S2: Without punctuation,121.0,63.0,0.5207
S3: Final tokens,68.0,45.0,0.6618
S4: Porter-stemmed S3,68.0,43.0,0.6324


Top 10 words in S1:


,Word,Frequency
0,.,21
1,the,12
2,it,7
3,is,6
4,yesterday,5
5,i,3
6,not,3
7,since,3
8,this,3
9,works,3


Top 10 words in S3:


,Word,Frequency
0,yesterday,5
1,since,3
2,works,3
3,another,3
4,computer,3
5,python,2
6,working,2
7,check,2
8,vpn,2
9,stopped,2


**Answer:** TTR is the ratio of unique tokens to total tokens. From S1 to S4, punctuation and stopword removal change the token population, while Porter stemming can merge related forms such as different inflections into one stem. The exact direction and size of the change depend on this dataset's repeated words and word forms.

## Q6. Interpretation

### (a) Why does TTR change from S1 to S4?

Type-Token Ratio (TTR) is calculated by dividing the number of unique tokens by the total number of tokens. As preprocessing progresses from S1 to S4, punctuation and stopwords are removed, and stemming or lemmatization may combine different word forms into the same base form. Consequently, the total number of tokens and the number of unique tokens can change, causing the TTR to increase or decrease depending on the dataset.

### (b) Give one example where `word_tokenize()` and your custom tokenizer differ.

An example is an email address such as `it.helpdesk@thapar.edu`. The custom tokenizer is designed to preserve email addresses as single tokens, whereas `word_tokenize()` can split an email address into multiple tokens. This difference occurs because the custom regular expression explicitly recognizes email patterns.

### (c) Give one example where stemming produces an undesirable result.

The Q4 comparison table shows that the Lancaster Stemmer converts `student` to `stud`, `computer` to `comput`, and `error` to `er`. These results demonstrate over-stemming because the stemmer removes too many characters and produces incomplete word forms. For example, `stud` is not a complete dictionary form of `student`. This illustrates a limitation of aggressive stemming algorithms.

### (d) Give one example where POS-aware lemmatization preserves meaning better than stemming.

The Q4 table shows that the word `uploading` remains `uploading` after POS-aware lemmatization, while the Porter and Lancaster stemmers convert it to `upload`. This demonstrates that lemmatization uses linguistic information to determine a dictionary form, while stemming mechanically removes suffixes. However, because the table contains the word `uploading` without its sentence context, this example alone does not establish that the lemmatizer is more accurate. The correct lemma depends on the word's grammatical role in the sentence.